# Day 41 — Handling class imbalance
Objectives:
- Diagnose imbalance and choose metrics (PR AUC, recall, specificity).
- Techniques: class weights, resampling (SMOTE/undersample).
- Threshold tuning.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression
X,y = make_classification(n_samples=5000, weights=[0.95,0.05], flip_y=0.01, random_state=42)
Xtr,Xte,ytr,yte = train_test_split(X,y, stratify=y, random_state=42)
clf = LogisticRegression(max_iter=1000, class_weight='balanced').fit(Xtr,ytr)
pred = clf.predict(Xte)
proba = clf.predict_proba(Xte)[:,1]
print(classification_report(yte, pred, digits=3))
roc_auc_score(yte, proba)


### Optional: SMOTE
Note: imblearn may be needed: `pip install imbalanced-learn`

In [ ]:
# from imblearn.over_sampling import SMOTE
# sm = SMOTE(random_state=42)
# Xtr2, ytr2 = sm.fit_resample(Xtr, ytr)
# clf2 = LogisticRegression(max_iter=1000).fit(Xtr2,ytr2)
# roc_auc_score(yte, clf2.predict_proba(Xte)[:,1])


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — rare-class metrics, training interventions, and threshold policy

### Mental model

Class imbalance means the outcome of interest is uncommon, not that the
dataset is automatically unusable. Accuracy can look excellent when a
classifier always predicts the majority class. Start by recording
prevalence and the counts/costs of false negatives and false positives.

Class weights and resampling change the **training objective or training
distribution**. A decision threshold changes how fitted scores become
actions. These are separate choices. Resampling must occur inside each
training fold, and a threshold must be selected on validation evidence,
never final-test labels.

### Read the API before running it

- **`np.bincount(y)` / prevalence:** establishes class support before modeling and must be reported for every evaluation split.
- **`class_weight='balanced'`:** reweights training loss; it does not make the observed population balanced.
- **`precision_recall_curve(y, score)`:** shows threshold trade-offs; selecting a threshold still requires a decision rule and validation boundary.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — expose the majority-class accuracy trap

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** Positive cases are the operational focus; missing all of them is unacceptable.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, recall_score

y_true = np.array([0] * 95 + [1] * 5)
always_negative = np.zeros_like(y_true)
print({
    "prevalence": y_true.mean(),
    "accuracy": accuracy_score(y_true, always_negative),
    "positive_recall": recall_score(y_true, always_negative),
})
assert accuracy_score(y_true, always_negative) == 0.95
assert recall_score(y_true, always_negative) == 0.0

**Expected observation:** The classifier is 95% accurate while finding none of the positive cases.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — choose a threshold from an explicit recall constraint

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** The tiny array is a mechanics demo; a real threshold needs sufficient representative validation support and uncertainty.

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve

y_valid = np.array([0, 0, 0, 0, 1, 1, 1, 1])
scores = np.array([0.05, 0.15, 0.35, 0.55, 0.30, 0.50, 0.70, 0.90])
precision, recall, thresholds = precision_recall_curve(y_valid, scores)
candidates = [
    (threshold, p, r)
    for threshold, p, r in zip(thresholds, precision[:-1], recall[:-1])
    if r >= 0.75
]
chosen = max(candidates, key=lambda row: row[1])
print({"threshold": chosen[0], "precision": chosen[1], "recall": chosen[2]})

**Expected observation:** The chosen threshold satisfies the stated recall floor and maximizes precision only among eligible validation candidates.

### Debugging and practice ramp

**Common mistake:** Applying SMOTE or another sampler before the train/test split, which creates related synthetic evidence across boundaries.

**Diagnostic:** Trace row IDs through split, sampling, fitting, calibration, threshold selection, and final evaluation; print class counts at each stage.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define rare-class metrics, training interventions, and threshold policy in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not choose a threshold or sampler from final test results or claim synthetic rows add real-world evidence.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Compare `class_weight="balanced"` with a SMOTE strategy.

**Verify:** For task `Compare classweight="balanced" with a SMOTE strategy`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.






2. Tune the threshold to maximize minority-class F1.

**Verify:** For task `Tune the threshold to maximize minority-class F1`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then report class support and confusion counts at the chosen threshold and prove the declared operating constraint is satisfied.






3. Plot precision–recall curves and discuss the tradeoff.

**Verify:** For task `Plot precision–recall curves and discuss the tradeoff`, show the labeled figure and reconcile it with a numeric summary so appearance is not the only check; then report class support and confusion counts at the chosen threshold and prove the declared operating constraint is satisfied.







### Progressive hints

1. Resample only `X_train, y_train`. For cross-validation, place SMOTE inside an
   `imblearn.pipeline.Pipeline` so each fold synthesizes from its training rows.
2. Use `precision_recall_curve` on validation scores. Check array lengths
   carefully: thresholds have one fewer element than precision and recall.
3. Plot recall on the x-axis and precision on the y-axis; add class prevalence
   as a simple reference and label average precision.

### Additional mastery practice

Evaluate the minority outcome explicitly and keep resampling, calibration, threshold choice, and entity boundaries inside the correct training scope.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Prevalence-shift reasoning:** Hold sensitivity and specificity fixed while changing event prevalence from 20% to 2%. Predict how precision changes and verify it with Bayes' rule.
   **Progressive hint:** Precision depends on the base rate: TP/(TP+FP). Use a hypothetical population such as 10,000 to make the counts visible.

**Verify:** For task `Prevalence-shift reasoning: Hold sensitivity and specificity fixed while changing event preva...`, state one precise claim, the evidence supporting it, the governing assumption, and a counterexample or limitation; then report class support and confusion counts at the chosen threshold and prove the declared operating constraint is satisfied.







5. **Grouped imbalance split:** Create a cross-validation plan for rare outcomes with multiple rows per account. Assert both group separation and acceptable positive support in each fold.
   **Progressive hint:** Use StratifiedGroupKFold when feasible. Print group overlap, positive count, negative count, and prevalence per validation fold.

**Verify:** For task `Grouped imbalance split: Create a cross-validation plan for rare outcomes with multiple rows...`, produce the requested artifact with every named field/control and walk one allowed plus one rejected scenario through it; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.







6. **Calibration after resampling:** Explain why probabilities from a model trained on oversampled data may not match real prevalence. Design a calibration evaluation using unresampled validation data.
   **Progressive hint:** Oversampling changes the class distribution seen during fitting. Fit/calibrate inside development data and assess reliability on natural prevalence.

**Verify:** For task `Calibration after resampling: Explain why probabilities from a model trained on oversampled d...`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then produce the requested artifact with every named field/control and walk one allowed plus one rejected scenario through it.






Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Prevalence-shift reasoning


# Practice 5 — Grouped imbalance split


# Practice 6 — Calibration after resampling
